# Membre 2 - Streaming & Mobilité CityBikes

Ce notebook présente et lance le pipeline de collecte mobilité pour **Paris, Lyon et Lille**. Le code métier est partagé avec le script `scripts/citybikes_stream.py`, ce qui permet de lancer le même pipeline en continu hors de Jupyter.

Le pipeline produit :

1. **Bronze** : payloads JSON bruts partitionnés par réseau et date ;
2. **Silver** : stations typées en Parquet, avec timestamp UTC et déduplication ;
3. **Gold mobilité** : dernier état par station et indicateurs horaires.

## 1. Paramétrage

La fréquence de collecte retenue est de 5 minutes, soit 300 secondes. Cette cadence limite aussi le risque de rate limiting CityBikes.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'cities.yml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.citybikes_stream import (
    TARGET_CITIES,
    build_session,
    collect_once,
    build_minio_client,
    ensure_bucket,
    load_cities,
    read_silver,
    read_silver_remote,
)

CONFIG_PATH = PROJECT_ROOT / 'config' / 'cities.yml'
DATA_ROOT = PROJECT_ROOT / 'data'
cities = load_cities(CONFIG_PATH)
TIMEOUT = (5, 15)
REQUEST_DELAY_SECONDS = 1.0
STORAGE_MODE = 'minio'  # 'minio' : tout en ligne (R2/MinIO) ; 'local' : tout en local, sans cloud
minio_client = None
minio_buckets = None
if STORAGE_MODE != 'local':
    minio_client, minio_buckets = build_minio_client()
    for bucket in minio_buckets.values():
        ensure_bucket(minio_client, bucket)

print('Villes :', [cities[city_id]['name'] for city_id in TARGET_CITIES])
print('Réseaux :', {city_id: cities[city_id]['citybikes_network_id'] for city_id in TARGET_CITIES})
print('Sortie :', DATA_ROOT)

Villes : ['Paris', 'Lyon', 'Lille']
Réseaux : {'paris': 'velib', 'lyon': 'velov', 'lille': 'vlille'}
Sortie : data


## 2. Collecte d'un micro-batch

Cette cellule lance un seul cycle. Pour le streaming récurrent, utiliser le script depuis un terminal afin de ne pas bloquer le notebook. Une erreur sur une ville est journalisée et n'empêche pas les autres villes d'être traitées.

In [2]:
session = build_session()
collected_rows = collect_once(
    session=session,
    cities=cities,
    output_root=DATA_ROOT,
    timeout=TIMEOUT,
    request_delay=REQUEST_DELAY_SECONDS,
    minio_client=minio_client,
    minio_buckets=minio_buckets,
)
print(f'Lignes collectées pendant ce cycle : {collected_rows:,}')

Lignes collectées pendant ce cycle : 2,247


## 3. Vérification de la Silver

La clé logique de déduplication est `(network_id, station_id, snapshot_minute_utc)`. Elle évite de compter deux fois une relance dans la même minute tout en conservant les observations successives.

In [3]:
if minio_client is not None:
    # En ligne : on relit le Silver publié (R2/MinIO), pas celui de cette machine.
    silver = read_silver_remote(minio_client, minio_buckets['silver'])
else:
    silver = read_silver(DATA_ROOT)
if silver.empty:
    print('Aucune donnée Silver disponible. Vérifier la connectivité ou le rate limiting CityBikes.')
else:
    print(f'{len(silver):,} lignes Silver')
    display(silver.groupby(['city_id', 'city_name'], as_index=False).agg(
        stations=('station_id', 'nunique'),
        observations=('station_id', 'count'),
        dernier_releve=('observed_at_utc', 'max'),
    ))
    display(silver.head())

164,031 lignes Silver


,city_id,city_name,stations,observations,dernier_releve
0,lille,Lille,267,19491,2026-11-20T15:32:06.099299+00:00
1,lyon,Lyon,477,33434,2026-11-20T15:32:04.501997+00:00
2,paris,Paris,1522,111106,2026-11-20T15:32:02.436958+00:00


,city_id,city_name,network_id,station_id,station_name,latitude,longitude,bikes_available,free_slots,station_timestamp,station_status,observed_at_utc,snapshot_minute_utc
0,paris,Paris,velib,0003131bffcf527ba3a560fc46d38b6f,Argenson - Château,48.888559,2.264200,17,8,2026-11-19T11:05:15.701711+00:00Z,None,2026-11-19T11:09:44.416627+00:00,2026-11-19T11:09:00+00:00
1,paris,Paris,velib,000471e1f2b458a6e68a8ae732880557,Saint Lambert - Blomet,48.836591,2.293056,2,24,2026-11-19T11:05:15.711273+00:00Z,None,2026-11-19T11:09:44.416627+00:00,2026-11-19T11:09:00+00:00
2,paris,Paris,velib,001e872d5c8bb1d9d69593da89a6da9e,Saint-Maur - République,48.864186,2.378317,8,42,2026-11-19T11:05:15.708236+00:00Z,None,2026-11-19T11:09:44.416627+00:00,2026-11-19T11:09:00+00:00
3,paris,Paris,velib,0028fad7cbbbe461aa18ae720500f8ef,Mairie de Rosny-sous-Bois,48.871257,2.486581,23,6,2026-11-19T11:05:15.702777+00:00Z,None,2026-11-19T11:09:44.416627+00:00,2026-11-19T11:09:00+00:00
4,paris,Paris,velib,003fde84c86105536fa7bc92ce5bd8a9,Labouret - Saint-Denis,48.924067,2.253916,19,5,2026-11-19T11:05:15.705783+00:00Z,None,2026-11-19T11:09:44.416627+00:00,2026-11-19T11:09:00+00:00


## 4. Indicateurs Gold mobilité

Les KPI sont calculés à partir de la Silver dédupliquée et écrits dans les deux tables attendues.

**Source de vérité :** en ligne (`STORAGE_MODE = 'minio'`), la Gold est recalculée depuis la Silver **publiée** (R2/MinIO), sur les deux dernières heures, puis fusionnée avec la Gold publiée ; le Silver local de cette machine n'y entre jamais. En mode `local`, tout reste local.

Définitions :

- `mobility_station_latest.parquet` : état le plus récent de chaque station ;
- `mobility_hourly.parquet` : stocks moyens/totaux, taux de rupture, taux de saturation et stations critiques ;
- les taux sont calculés sur les observations réellement collectées.

La clé Silver de déduplication est `(network_id, station_id, snapshot_minute_utc)`.

In [4]:
gold_dir = DATA_ROOT / 'gold' / 'kpis'
latest_path = gold_dir / 'mobility_station_latest.parquet'
hourly_path = gold_dir / 'mobility_hourly.parquet'
if latest_path.exists() and hourly_path.exists():
    mobility_latest = pd.read_parquet(latest_path)
    mobility_hourly = pd.read_parquet(hourly_path)
    print(f'mobility_station_latest : {len(mobility_latest):,} lignes')
    print(f'mobility_hourly        : {len(mobility_hourly):,} lignes')
    display(mobility_hourly.tail(10))
else:
    print('Les tables Gold seront disponibles après une collecte réussie.')

mobility_station_latest : 2,266 lignes
mobility_hourly        : 27 lignes


,city_id,city_name,hour_utc,stations_observed,observations,bikes_available_total,bikes_available_mean,free_slots_total,free_slots_mean,valid_bike_observations,valid_slot_observations,empty_observations,full_observations,breakdown_rate,saturation_rate,critical_station_count,critical_station_rate
17,lyon,Lyon,2026-11-20 15:00:00+00:00,458,3206,15445,4.81753,45314,14.134124,3206,3206,759,73,0.236744,0.022770,212,0.462882
18,paris,Paris,2026-11-19 11:00:00+00:00,1522,3044,38926,12.787779,58292,19.149803,3044,3044,305,135,0.100197,0.044350,309,0.203022
19,paris,Paris,2026-11-19 16:00:00+00:00,1522,4566,56423,12.357205,89359,19.570521,4566,4566,321,122,0.070302,0.026719,184,0.120894
20,paris,Paris,2026-11-20 09:00:00+00:00,1522,15220,192772,12.665703,291873,19.176938,15220,15220,1457,640,0.095729,0.042050,335,0.220105
21,paris,Paris,2026-11-20 10:00:00+00:00,1522,18264,232714,12.741678,349610,19.142028,18264,18264,1696,855,0.092860,0.046813,353,0.231932
22,paris,Paris,2026-11-20 11:00:00+00:00,1522,7610,95730,12.579501,147046,19.322733,7610,7610,669,293,0.087911,0.038502,270,0.177398
23,paris,Paris,2026-11-20 12:00:00+00:00,1522,16742,210423,12.56857,323797,19.340401,16742,16742,1340,649,0.080038,0.038765,352,0.231275
24,paris,Paris,2026-11-20 13:00:00+00:00,1522,18264,231919,12.698149,351106,19.223938,18264,18264,1371,675,0.075066,0.036958,320,0.210250
25,paris,Paris,2026-11-20 14:00:00+00:00,1522,16742,212268,12.678772,322475,19.261438,16742,16742,1306,568,0.078007,0.033927,321,0.210907
26,paris,Paris,2026-11-20 15:00:00+00:00,1522,10654,133013,12.484794,207424,19.46912,10654,10654,777,316,0.072930,0.029660,254,0.166886


## 5. Lancement streaming hors notebook

Depuis la racine du projet :

```bash
uv run python scripts/citybikes_stream.py
```

L'intervalle est de 300 secondes (5 minutes), choix de l'équipe ; l'énoncé indique 2 à 3 minutes. Ne pas descendre sous 3 minutes : l'API répond `HTTP 429` au-delà de son quota.

Tout en local, sans cloud :

```bash
uv run python scripts/citybikes_stream.py --storage local
```

Reconstruire la Gold uniquement depuis la Silver publiée (sans collecter), par exemple après un test local :

```bash
uv run python scripts/citybikes_stream.py --rebuild-gold
```

Arrêt propre avec `Ctrl+C`. Les fichiers Bronze, Silver et Gold déjà écrits sont conservés.